In [39]:
import requests
import pandas as pd

from datetime import date, timedelta

base_url = "https://eservices.mas.gov.sg/apimg-gw/server/monthly_statistical_bulletin_non610ora/exchange_rates_average_for_period_weekly/views/exchange_rates_average_for_period_weekly"
headers = {
    "accept": "application/json; charset=UTF-8",
    "KeyId": "dd277b45-55c5-4c1d-8d5e-2937466f8bf6"
    }
today = date.today()

for i in range(14):
    check_date = today - timedelta(days=i)
    end_of_week_param = check_date.strftime("%Y-%m-%d")

    params = {
        "end_of_week": end_of_week_param
    }

    response = requests.get(base_url, headers=headers, params=params)

    if response.status_code != 200:
        continue

    try:
        data = response.json()
    except:
        continue

    elements = data.get("elements", [])

    if elements:
        print("Latest available date found:", end_of_week_param)
        df = pd.DataFrame(elements)
        selected_columns = [
    "end_of_week",
    "aud_sgd", 
    "chf_sgd", 
    "cny_sgd_100",
    "eur_sgd",
    "gbp_sgd",
    "jpy_sgd_100",
    "usd_sgd",
    ]

        df_selected = df[selected_columns]

        # Convert all rate columns to numeric first
        rate_columns = selected_columns[1:]
        df_selected[rate_columns] = df_selected[rate_columns].apply(pd.to_numeric, errors="coerce")

        # Convert per 100 currency rates to per 1 unit
        df_selected["cny_sgd_100"] = df_selected["cny_sgd_100"].round(6) * 0.01
        df_selected["jpy_sgd_100"] = df_selected["jpy_sgd_100"].round(6) * 0.01

        print(df_selected)
        break
else:
    raise Exception("No exchange rate data found in the last 7 days.")

Latest available date found: 2026-05-29
  end_of_week  aud_sgd  chf_sgd  cny_sgd_100  eur_sgd  gbp_sgd  jpy_sgd_100  \
0  2026-05-29   0.9136   1.6284       0.1885   1.4857   1.7179     0.008028   

   usd_sgd  
0   1.2781  


C:\Users\Emily\AppData\Local\Temp\ipykernel_17724\3010618201.py:51: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected[rate_columns] = df_selected[rate_columns].apply(pd.to_numeric, errors="coerce")
C:\Users\Emily\AppData\Local\Temp\ipykernel_17724\3010618201.py:54: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected["cny_sgd_100"] = df_selected["cny_sgd_100"].round(6) * 0.01
C:\Users\Emily\AppData\Local\Temp\ipykernel_17724\3010618201.py:55: SettingWithCopyWarning: 
A value is trying to be set

In [40]:
df_selected = df_selected.rename(columns={
    "end_of_week": "Date",
    "eur_sgd": "EUR",
    "gbp_sgd": "GBP",
    "usd_sgd": "USD",
    "aud_sgd": "AUD",
    "cny_sgd_100": "CNY",
    "jpy_sgd_100": "JPY",
    "chf_sgd": "CHF"
})

date_value = pd.to_datetime(df_selected["Date"].iloc[0], errors="coerce") + pd.Timedelta(days=3)

currency_df = pd.DataFrame({
    "Currency": [df_selected.columns[i] for i in range(1, len(df_selected.columns))],
    "date": date_value.strftime("%m/%d/%Y"),
    "Rate": [df_selected.iloc[0, i] for i in range(1, len(df_selected.columns))]
})
print(currency_df)

  Currency        date      Rate
0      AUD  06/01/2026  0.913600
1      CHF  06/01/2026  1.628400
2      CNY  06/01/2026  0.188500
3      EUR  06/01/2026  1.485700
4      GBP  06/01/2026  1.717900
5      JPY  06/01/2026  0.008028
6      USD  06/01/2026  1.278100


In [41]:
# Duplicate for 7 days
date_value = pd.to_datetime(df_selected["Date"].iloc[0], errors="coerce") + pd.Timedelta(days=3)
currency_toupload = []

for d in pd.date_range(start=date_value, periods=7, freq="D"):
        temp_df = pd.DataFrame({
            "Currency": [df_selected.columns[j] for j in range(1, len(df_selected.columns))],
            "date": d.strftime("%m/%d/%Y"),
            "Rate": [df_selected.iloc[0, j] for j in range(1, len(df_selected.columns))]
        })
        currency_toupload.append(temp_df)

currency_toupload = pd.concat(currency_toupload, ignore_index=True)        
print(currency_toupload)


   Currency        date      Rate
0       AUD  06/01/2026  0.913600
1       CHF  06/01/2026  1.628400
2       CNY  06/01/2026  0.188500
3       EUR  06/01/2026  1.485700
4       GBP  06/01/2026  1.717900
5       JPY  06/01/2026  0.008028
6       USD  06/01/2026  1.278100
7       AUD  06/02/2026  0.913600
8       CHF  06/02/2026  1.628400
9       CNY  06/02/2026  0.188500
10      EUR  06/02/2026  1.485700
11      GBP  06/02/2026  1.717900
12      JPY  06/02/2026  0.008028
13      USD  06/02/2026  1.278100
14      AUD  06/03/2026  0.913600
15      CHF  06/03/2026  1.628400
16      CNY  06/03/2026  0.188500
17      EUR  06/03/2026  1.485700
18      GBP  06/03/2026  1.717900
19      JPY  06/03/2026  0.008028
20      USD  06/03/2026  1.278100
21      AUD  06/04/2026  0.913600
22      CHF  06/04/2026  1.628400
23      CNY  06/04/2026  0.188500
24      EUR  06/04/2026  1.485700
25      GBP  06/04/2026  1.717900
26      JPY  06/04/2026  0.008028
27      USD  06/04/2026  1.278100
28      AUD  0

In [42]:
!pip install openpyxl
from openpyxl import load_workbook

    # Duplicate the first data row
first_row = currency_toupload.iloc[[0]]

currency_toupload_final = pd.concat(
    [first_row, currency_toupload],
    ignore_index=True
)
txtfile_path = r"C:\Users\Emily\Desktop\Query\ExchangeRate_UplodtoSAP.txt"

currency_toupload_final.to_csv(txtfile_path, index=False, header=True, sep="\t")


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
